In [1]:
# Cell 1: Setup & Project Path
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip

# สร้าง Spark Session ด้วยค่าคอนฟิกที่ถูกต้องตามคู่มือ Delta Lake
builder = SparkSession.builder \
    .appName("Silver to Gold - SCD Type 2") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# ใช้ configure_spark_with_delta_pip เพื่อโหลด JARs อัตโนมัติ
spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/02/20 14:53:16 WARN Utils: Your hostname, RATCHANON-PAN.local resolves to a loopback address: 127.0.0.1; using 10.24.10.74 instead (on interface en0)
26/02/20 14:53:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/ratchanon.pan/.ivy2/cache
The jars for the packages stored in: /Users/ratchanon.pan/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-526f3e9f-8cdb-498a-a74e-0fea7352fd3a;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/ratchanon.pan/ecommerce-analytics/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 120ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-526f3e9f-8cdb-498a-a74e-0fea7352fd3a
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/5ms)
26/02/20 14:53:16

In [ ]:
# ฟังก์ชันช่วยหา Project Root เพื่อแก้ปัญหา PathNotFound
def get_project_root():
    path = os.path.abspath(os.getcwd())
    while os.path.basename(path) != 'ecommerce-analytics':
        new_path = os.path.dirname(path)
        if new_path == path: break
        path = new_path
    return path

project_root = get_project_root()

def get_bronze_path(file_name):
    return os.path.join(project_root, "data", "bronze", "olist", file_name)

def get_silver_path(filename):
    return os.path.join(project_root, "data", "silver", filename)

def get_gold_path(filename):
    return os.path.join(project_root, "data", "gold", filename)

In [ ]:
# Read Silver layers
df_customers_silver = spark.read.format("delta").load(get_silver_path("customers"))
df_orders_silver = spark.read.format("delta").load(get_silver_path("orders"))

print(f"Customers: {df_customers_silver.count()}")
print(f"Orders: {df_orders_silver.count()}")

26/02/20 14:53:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Customers: 99441
Orders: 99441


In [25]:
# Cell 3: Create dim_customer with SCD Type 2
# Add SCD Type 2 columns
df_dim_customer = df_customers_silver \
    .withColumn("surrogate_key", monotonically_increasing_id()) \
    .withColumn("effective_date", current_timestamp()) \
    .withColumn("end_date", lit("9999-12-31").cast("timestamp")) \
    .withColumn("is_current", lit(True)) \
    .select(
        "surrogate_key",
        col("customer_id").alias("customer_natural_key"),
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "effective_date",
        "end_date",
        "is_current"
    )

# Write initial load
df_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(get_gold_path("dim_customer"))

print("✅ dim_customer created with SCD Type 2 structure")

26/02/20 15:55:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ dim_customer created with SCD Type 2 structure


In [11]:
# Cell 4: Simulate SCD Type 2 Update
# สมมติมี customer ย้ายที่อยู่ (แก้ zip code)
from delta.tables import DeltaTable

# Read existing dimension
dim_customer = DeltaTable.forPath(spark, get_gold_path("customers"))

# Simulated new data (customer ย้ายบ้าน)
sample_customer = df_customers_silver.limit(1).collect()[0]
updated_customer = spark.createDataFrame([
    (
        sample_customer["customer_id"],
        sample_customer["customer_unique_id"],
        99999,  # new zip code
        "New City",
        sample_customer["customer_state"]
    )
], ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"])

updated_customer = updated_customer \
    .withColumn("effective_date", current_timestamp()) \
    .withColumn("end_date", lit("9999-12-31").cast("timestamp")) \
    .withColumn("is_current", lit(True))

In [12]:
df_customers_silver.show(5)

+--------------------+--------------------+------------------------+-------------+--------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|      ingestion_date|
+--------------------+--------------------+------------------------+-------------+--------------+--------------------+
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|  itapecerica|            MG|2026-02-20 11:42:...|
|000e943451fc2788c...|d73c3cf4a0922ece1...|                   99460|     colorado|            RS|2026-02-20 11:42:...|
|000fd45d6fedae68f...|cee6fa72fb403ef95...|                   12970|     piracaia|            SP|2026-02-20 11:42:...|
|001226b2341ef6204...|e7897290aea0805ab...|                   90470| porto alegre|            RS|2026-02-20 11:42:...|
|0013280441d86a4f7...|06caeba6db23a17bf...|                    5409|    sao paulo|            SP|2026-02-20 11:42:...|
+--------------------+--------------------+-----

In [16]:
# Cell 5: SCD Type 2 Merge Logic
dim_customer.alias("target").merge(
    updated_customer.alias("source"),
    "target.customer_natural_key = source.customer_id AND target.is_current = true"
).whenMatchedUpdate(
    condition = """
        target.customer_zip_code_prefix <> source.customer_zip_code_prefix OR
        target.customer_city <> source.customer_city
    """,
    set = {
        "end_date": current_timestamp(),
        "is_current": lit(False)
    }
).execute()

In [ ]:
# Insert new record with updated values
new_records = updated_customer \
    .join(
        dim_customer.toDF().filter(col("is_current") == False),
        updated_customer["customer_id"] == dim_customer.toDF()["customer_natural_key"],
        "inner"
    ) \
    .select(
        (col("surrogate_key") + 1000000).alias("surrogate_key"),
        col("customer_id").alias("customer_natural_key"),
        updated_customer["customer_unique_id"],
        updated_customer["customer_zip_code_prefix"],
        updated_customer["customer_city"],
        updated_customer["customer_state"],
        updated_customer["effective_date"],
        updated_customer["end_date"],
        updated_customer["is_current"]
    )

In [ ]:
from pyspark.sql.types import IntegerType

# Helper function — cast zip code ให้ consistent ก่อน write ทุกครั้ง
def fix_schema(df):
    return df.withColumn(
        "customer_zip_code_prefix",
        col("customer_zip_code_prefix").cast(IntegerType())
    )

# ใช้กับ new_records ก่อน write
new_records = fix_schema(new_records)

new_records.write \
    .format("delta") \
    .mode("append") \
    .save(get_gold_path("customers"))
print("✅ SCD Type 2 merge completed")

In [22]:
# Cell 6: Create dim_product (SCD Type 1 - simpler)
df_products_silver = spark.read.csv(
    get_bronze_path("olist_products_dataset.csv"),
    header=True,
    inferSchema=True
)

df_dim_product = df_products_silver \
    .withColumn("surrogate_key", monotonically_increasing_id()) \
    .select(
        "surrogate_key",
        col("product_id").alias("product_natural_key"),
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )

df_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save(get_gold_path("dim_product"))

print("✅ dim_product created")

✅ dim_product created


In [23]:
# Cell 7: Create fact_orders
df_fact_orders = df_orders_silver \
    .join(df_dim_customer, 
          df_orders_silver["customer_id"] == df_dim_customer["customer_natural_key"],
          "left") \
    .select(
        df_orders_silver["order_id"],
        col("surrogate_key").alias("customer_key"),
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_timestamp"
    ) \
    .withColumn("order_date_key", date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))

df_fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date_key") \
    .save(get_gold_path("fact_orders"))

print("✅ fact_orders created with partitioning")

✅ fact_orders created with partitioning


In [26]:
# Cell 8: Verify SCD Type 2
print("\n=== SCD Type 2 Verification ===")
df_dim = spark.read.format("delta").load(get_gold_path("dim_customer"))

# Show history for one customer
sample_id = df_dim.select("customer_natural_key").first()[0]
df_dim.filter(col("customer_natural_key") == sample_id) \
    .orderBy("effective_date") \
    .show(truncate=False)

print(f"\nTotal customer records: {df_dim.count()}")
print(f"Current records: {df_dim.filter(col('is_current') == True).count()}")
print(f"Historical records: {df_dim.filter(col('is_current') == False).count()}")


=== SCD Type 2 Verification ===
+-------------+--------------------------------+--------------------------------+------------------------+-------------+--------------+--------------------------+-------------------+----------+
|surrogate_key|customer_natural_key            |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|effective_date            |end_date           |is_current|
+-------------+--------------------------------+--------------------------------+------------------------+-------------+--------------+--------------------------+-------------------+----------+
|0            |0001fd6190edaaf884bcaf3d49edf079|94b11d37cd61cb2994a194d11f89682b|29830                   |nova venecia |ES            |2026-02-20 15:55:42.869929|9999-12-31 00:00:00|true      |
+-------------+--------------------------------+--------------------------------+------------------------+-------------+--------------+--------------------------+-------------------+---------